In [1]:
import pyomo.environ as pe
import pyomo.opt as po
solver = po.SolverFactory('appsi_highs')
model = pe.ConcreteModel()

In [ ]:
processing_times=[
  [3,3,5],
  [2,5,6],
  [3,2,5]
]
profit_per_product = [30,35,32]
max_processing_time = [600,800,1200]

amount_machines=3
amount_products=3

In [ ]:
model.J = pe.RangeSet(1,amount_machines)
model.I = pe.RangeSet(1,amount_products)
model.p = pe.Param(model.I, initialize=lambda m, i: profit_per_product[i-1])
model.h = pe.Param(model.J,initialize=lambda m, j: max_processing_time[j-1])
model.a = pe.Param(model.I, model.J, initialize=lambda m, i, j: processing_times[i-1][j-1])

In [4]:
model.x=pe.Var(model.I,domain=pe.NonNegativeReals)

In [5]:
obj_expr=sum(model.p[i]*model.x[i] for i in model.I)
model.obj=pe.Objective(sense=pe.maximize, expr=obj_expr)

In [6]:
def con_expr1(model, j):
    return sum(model.a[i,j] * model.x[i] for i in model.I) <= model.h[j] 
model.con1 = pe.Constraint(model.J, rule=con_expr1)
result = solver.solve(model)

In [7]:
model.display()

Model unknown

  Variables:
    x : Size=3, Index=I
        Key : Lower : Value              : Upper : Fixed : Stale : Domain
          1 :     0 :                0.0 :  None : False : False : NonNegativeReals
          2 :     0 :  74.99999999999996 :  None : False : False : NonNegativeReals
          3 :     0 : 150.00000000000006 :  None : False : False : NonNegativeReals

  Objectives:
    obj : Size=1, Index=None, Active=True
        Key  : Active : Value
        None :   True : 7425.0

  Constraints:
    con1 : Size=3
        Key : Lower : Body              : Upper
          1 :  None : 600.0000000000001 :  600.0
          2 :  None : 674.9999999999999 :  800.0
          3 :  None :            1200.0 : 1200.0


In [8]:
for i in model.I:
  print(f"x{i}:",pe.value(model.x[i]))
print("Goal:",pe.value(model.obj))

x1: 0.0
x2: 74.99999999999996
x3: 150.00000000000006
Goal: 7425.0


## QMKP

In [1]:
import pyomo.environ as pe
import pyomo.opt as po
solver = po.SolverFactory('gurobi_direct')
model = pe.ConcreteModel()

In [2]:
import random
#parameter declarations
n=6 #resources
m=2 #projects
print("\cost resources")
w=[]
for i in range(n):
    w.append(random.randint(10, 50))
print(w)
#profit for n items
print("\nvalue per resource")
p = []
for i in range(n):
    p.append(random.randint(10, 50)*10)
print(p)
# capacity of m projects
c = [100, 50]
# synergy for every pair of resource
print("\nsynergy per pair of resource:")
q = []
for i in range(n):
    qq = []
    for j in range(n):
        qq.append(0) # synergy of the same employee (diagonally) is 0
    q.append(qq)
for i in range(n-1):
    # q[i][i] = random.randint(10, 50)*10
    for j in range(i+1, n):
        q[i][j] = random.randint(10, 50) * 10
        q[j][i] = q[i][j]
print(q)

\cost resources
[30, 12, 41, 22, 30, 22]

value per resource
[390, 180, 430, 380, 250, 440]

synergy per pair of resource:
[[0, 500, 450, 330, 240, 290], [500, 0, 230, 180, 330, 500], [450, 230, 0, 350, 120, 120], [330, 180, 350, 0, 360, 410], [240, 330, 120, 360, 0, 200], [290, 500, 120, 410, 200, 0]]


In [3]:
model.n = pe.RangeSet(1,n)
model.m = pe.RangeSet(1,m)
model.w = pe.Param(model.n, initialize=lambda m, i: w[i-1])
model.p = pe.Param(model.n,initialize=lambda m, j: p[j-1])
model.c = pe.Param(model.m,initialize=lambda m, j: c[j-1])
model.q = pe.Param(model.n, model.n, initialize=lambda m, i, j: q[i-1][j-1])

In [4]:
model.x=pe.Var(model.n,model.m,domain=pe.Binary)

In [5]:
obj_expr = sum(
    # First part: Sum of individual profits
    sum(model.p[i] * model.x[i, j] for i in model.n) +
    # Second part: Sum of synergy profits for unique pairs
    sum(model.q[i, k] * model.x[i, j] * model.x[k, j] for i in model.n for k in model.n if i < k)
    # Outer summation over all projects
    for j in model.m
)

model.obj = pe.Objective(sense=pe.maximize, expr=obj_expr)

In [6]:
def con_expr1(model, j):
    return sum(model.w[i] * model.x[i,j] for i in model.n) <= model.c[j] 
model.con1 = pe.Constraint(model.m, rule=con_expr1)

def con_expr2(model, i):
    return sum(model.x[i,j] for j in model.m) <= 1
model.con2 = pe.Constraint(model.n, rule=con_expr2)
result = solver.solve(model)

In [7]:
model.display()

Model unknown

  Variables:
    x : Size=12, Index=n*m
        Key    : Lower : Value : Upper : Fixed : Stale : Domain
        (1, 1) :     0 :   1.0 :     1 : False : False : Binary
        (1, 2) :     0 :  -0.0 :     1 : False : False : Binary
        (2, 1) :     0 :   1.0 :     1 : False : False : Binary
        (2, 2) :     0 :  -0.0 :     1 : False : False : Binary
        (3, 1) :     0 :  -0.0 :     1 : False : False : Binary
        (3, 2) :     0 :   1.0 :     1 : False : False : Binary
        (4, 1) :     0 :   1.0 :     1 : False : False : Binary
        (4, 2) :     0 :  -0.0 :     1 : False : False : Binary
        (5, 1) :     0 :  -0.0 :     1 : False : False : Binary
        (5, 2) :     0 :  -0.0 :     1 : False : False : Binary
        (6, 1) :     0 :   1.0 :     1 : False : False : Binary
        (6, 2) :     0 :  -0.0 :     1 : False : False : Binary

  Objectives:
    obj : Size=1, Index=None, Active=True
        Key  : Active : Value
        None :   True : 40